In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import MaxNLocator

R_GAS = 1.987204258e-3  # Gas constant kcal/(mol*K)


def convert_r_rho(r):
    k = 0.108
    l0 = 14.24
    n = 8
    numerator = r * (1 - 1 / (1 + (k * l0 * r) ** n) ** 2)
    denominator = 1 - 1 / (1 + (k * l0) ** n) ** 2
    return numerator / denominator



def convert_rho_to_r(rho):
    r_grid = np.linspace(0.0, 1.0, 20001)
    rho_grid = convert_r_rho(r_grid)
    rho_values = np.asarray(rho, dtype=float)
    rho_clipped = np.clip(rho_values, rho_grid.min(), rho_grid.max())
    return np.interp(rho_clipped, rho_grid, r_grid)



def prop(r, ur):
    k = 0.108
    l0 = 14.24
    ul0 = 0.08
    n = 8
    kl0 = k * l0
    kl0_r = kl0 * r
    term_kl0_n = (1 + kl0 ** n)
    term_kl0r_n = (1 + kl0_r ** n)

    num1 = (2 * k * n * r**2 * kl0_r**(n - 1)) / ((1 - 1 / term_kl0_n**2) * term_kl0r_n**3)
    num2 = (2 * k * kl0**(n - 1) * n * r * (1 - 1 / term_kl0r_n**2)) / ((1 + kl0**n)**3 * (1 - 1 / term_kl0_n**2)**2)
    term_ul0 = num1 - num2

    num3 = (2 * kl0 * n * r * kl0_r**(n - 1)) / ((1 - 1 / term_kl0_n**2) * term_kl0r_n**3)
    num4 = (1 - 1 / term_kl0r_n**2) / (1 - 1 / term_kl0_n**2)
    term_ur = num3 + num4

    uncert_ul0 = term_ul0**2 * ul0**2
    uncert_ur = term_ur**2 * ur**2
    return np.sqrt(uncert_ul0 + uncert_ur)



def truncate_to_first_significant(value, uncertainty):
    if uncertainty == 0:
        return value
    decimal_places = -int(np.floor(np.log10(abs(uncertainty))))
    factor = 10 ** decimal_places
    return np.round(value * factor) / factor



def truncate_df_to_uncertainty(df, target_column, uncertainty_column):
    df[target_column] = df.apply(lambda row: truncate_to_first_significant(row[target_column], row[uncertainty_column]), axis=1)
    df[uncertainty_column] = df[uncertainty_column].apply(lambda x: truncate_to_first_significant(x, x))
    return df



def f_PA(tT, c0, c1, c2):
    t, T = tT
    return c0 + c1 * np.log(t) + c2 / (R_GAS * T)



def lnt_PA(T, rho, params):
    c0, c1, c2 = params
    return np.log(1 - rho) / c1 - (c2 / c1) / (R_GAS * T) - c0 / c1



def f_PC(tT, c0, c1, c2):
    t, T = tT
    return c0 + c1 * np.log(t) + c2 * np.log(1 / (R_GAS * T))



def lnt_PC(T, rho, params):
    c0, c1, c2 = params
    return np.log(1 - rho) / c1 - (c2 / c1) * np.log(1 / (R_GAS * T)) - c0 / c1



def f_FA(tT, c0, c1, c2, c3):
    t, T = tT
    return c0 + c1 * (np.log(t) - c2) / (1 / (R_GAS * T) - c3)



def lnt_FA(T, rho, params):
    c0, c1, c2, c3 = params
    numerator = c1 * c2 * R_GAS * T + c0 * (-1 + c3 * R_GAS * T) + (1 - c3 * R_GAS * T) * np.log(1 - rho)
    denominator = c1 * R_GAS * T
    return numerator / denominator



def f_FC(tT, c0, c1, c2, c3):
    t, T = tT
    return c0 + c1 * (np.log(t) - c2) / (np.log(1 / (R_GAS * T)) - c3)



def lnt_FC(T, rho, params):
    c0, c1, c2, c3 = params
    return (c1 * c2 + c0 * c3 - c3 * np.log(1 - rho) + np.log(1 / (R_GAS * T)) * (-c0 + np.log(1 - rho))) / c1



def load_and_process_data(file_path):
    df = pd.read_csv(file_path, sep=r"\s+", engine='python')
    df = truncate_df_to_uncertainty(df, 'r', 'e.r')
    processed = df.iloc[:, [4, 5, 6, 7]].copy()
    processed.loc[:, 'rho'] = convert_r_rho(df['r'])
    processed.loc[:, 'e.rho'] = prop(df['r'], df['e.r'])
    processed = truncate_df_to_uncertainty(processed, 'rho', 'e.rho')
    return processed



def DAM_fit(model_function, initial_params, fit_variable='rho'):
    t_data = dataDurango['tempoS']
    T_data = dataDurango['temperK']

    if fit_variable == 'r':
        target_data = dataDurango['r']
        sigma_f = dataDurango['e.r']
    else:
        target_data = dataDurango['rho']
        sigma_f = dataDurango['e.rho']

    def target_model(tT, *params):
        predicted_rho = 1 - np.exp(model_function(tT, *params))
        if fit_variable == 'r':
            return convert_rho_to_r(predicted_rho)
        return predicted_rho

    params, covariance = curve_fit(
        target_model,
        (t_data, T_data),
        target_data,
        sigma=sigma_f,
        absolute_sigma=True,
        p0=initial_params
    )

    errors = np.sqrt(np.diag(covariance))
    fitted_values = target_model((t_data, T_data), *params)
    residuals = target_data - fitted_values

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((target_data - np.mean(target_data)) ** 2)
    r_squared = 1 - ss_res / ss_tot
    chi_squared = np.sum((residuals / sigma_f) ** 2)
    reduced_chi_squared = chi_squared / (len(target_data) - len(params))

    return {
        'params': params,
        'errors': errors,
        'covariance': covariance,
        'r_squared': r_squared,
        'chi_squared': chi_squared,
        'reduced_chi_squared': reduced_chi_squared,
        'fitted_values': fitted_values,
        'residuals': residuals,
        'sigma': sigma_f,
        'fit_variable': fit_variable,
    }



def analyze_residuals(df):
    residuals = df['residuals'].values
    sigma = df['sigma'].values
    fitted = df['fitted_values'].values
    norm_residuals = residuals / sigma

    print(f"Number of points: {len(residuals)}")
    print(f"Mean residual: {np.mean(residuals):.4f}")
    print(f"Std residual: {np.std(residuals):.4f}")
    print(f"Mean norm residual: {np.mean(norm_residuals):.4f}")
    print(f"Std norm residual: {np.std(norm_residuals):.4f}")

    outliers = np.where(np.abs(norm_residuals) > 3)[0]
    print(f"Outliers (>3 sigma): {len(outliers)} at indices {outliers}")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.hist(residuals, bins=20, color='blue', alpha=0.7)
    plt.title("Residuals Histogram")
    plt.xlabel("Residual")
    plt.ylabel("Frequency")

    plt.subplot(1, 2, 2)
    plt.hist(norm_residuals, bins=20, color='green', alpha=0.7)
    plt.title("Normalized Residuals Histogram")
    plt.xlabel("Normalized Residual")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.errorbar(fitted, residuals, yerr=sigma, fmt='o', color='red', ecolor='black', capsize=3, alpha=0.7)
    plt.axhline(0, linestyle='--', color='black')
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.grid(True, linestyle='--', linewidth=0.5)
    plt.show()



def plot_pseudo_arrhenius_dual(
    dDKTB,
    lnt_model_A, lnt_model_B,
    params_A, params_B,
    model_name_A, model_name_B,
    residuals_A, fitted_A, sigma_A,
    residuals_B, fitted_B, sigma_B
):
    Temps = np.linspace(293, 1000, 200)
    norm_residuals_A = residuals_A / sigma_A
    norm_residuals_B = residuals_B / sigma_B

    fig = plt.figure(figsize=(16, 6), constrained_layout=True)
    gs = gridspec.GridSpec(3, 3, figure=fig, width_ratios=[1, 2.0, 1], wspace=0)

    ax_main = fig.add_subplot(gs[:, 1])
    ax_resid_A = fig.add_subplot(gs[0, 0])
    ax_hist_A = fig.add_subplot(gs[1, 0])
    ax_qq_A = fig.add_subplot(gs[2, 0])
    ax_resid_B = fig.add_subplot(gs[0, 2])
    ax_hist_B = fig.add_subplot(gs[1, 2])
    ax_qq_B = fig.add_subplot(gs[2, 2])

    groups = [
        (dDKTB['rho'] > 0.9, '^', 'black', 'black', r'$\delta$ > 0.9'),
        ((dDKTB['rho'] > 0.8) & (dDKTB['rho'] <= 0.9), 'o', 'gray', 'black', r'0.8 < $\delta$ ≤ 0.9'),
        ((dDKTB['rho'] > 0.7) & (dDKTB['rho'] <= 0.8), 's', 'black', 'black', r'0.7 < $\delta$ ≤ 0.8'),
        ((dDKTB['rho'] > 0.6) & (dDKTB['rho'] <= 0.7), 'D', 'gray', 'black', r'0.6 < $\delta$ ≤ 0.7'),
    ]

    scatter_handles = []
    scatter_labels = []
    for cond, marker, facecolor, edgecolor, label in groups:
        subset = dDKTB.loc[cond]
        x = 1000 / subset['temperK']
        y = np.log(subset['tempoS'])
        sc = ax_main.scatter(x, y, s=50, c=facecolor, marker=marker, edgecolor=edgecolor, alpha=0.7, label=label)
        scatter_handles.append(sc)
        scatter_labels.append(label)

    line1 = ax_main.plot(1000 / Temps, lnt_model_A(Temps, 0.55, params_A), color='blue', linestyle='--', label=f'{model_name_A}: δ=0.55', linewidth=2)
    line2 = ax_main.plot(1000 / Temps, lnt_model_A(Temps, 0.9, params_A), color='blue', linestyle='--', label=f'{model_name_A}: δ=0.90', linewidth=2)
    line3 = ax_main.plot(1000 / Temps, lnt_model_B(Temps, 0.55, params_B), color='red', label=f'{model_name_B}: δ=0.55', linewidth=2)
    line4 = ax_main.plot(1000 / Temps, lnt_model_B(Temps, 0.9, params_B), color='red', label=f'{model_name_B}: δ=0.90', linewidth=2)

    ax_main.set_xlabel(r'1000 / T (K$^{-1}$)', fontsize=22)
    ax_main.set_ylabel(r'$\ln(t)$ (time in seconds)', fontsize=22)
    ax_main.set_xlim(0, 4)
    ax_main.set_ylim(-20, 50)
    ax_main.grid(True, linestyle='--', color='gray', linewidth=0.5)
    ax_main.tick_params(labelsize=16)

    legend_exp = ax_main.legend(handles=scatter_handles, labels=scatter_labels, loc='lower right', fontsize=15, title=r'Transformed $\delta$ exp. data ')
    legend_exp.get_title().set_fontsize(15)
    ax_main.add_artist(legend_exp)

    from matplotlib.lines import Line2D
    model_legend_handles = [
        Line2D([0], [0], color='blue', linestyle='--', lw=1, label=model_name_A),
        Line2D([0], [0], color='red', lw=1, label=model_name_B)
    ]
    legend_model = ax_main.legend(handles=model_legend_handles, loc='upper left', fontsize=15, title='Fitted Models')
    legend_model.get_title().set_fontsize(15)

    ax_main.text(2.0, 40, 'δ = 0.55', color='black', fontsize=15, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    ax_main.text(3.0, 25, 'δ = 0.90', color='black', fontsize=15, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    title_style = dict(fontsize=14, pad=6)

    ax_resid_A.errorbar(fitted_A, residuals_A, yerr=sigma_A, fmt='o', color='blue', ecolor='black', capsize=3, alpha=0.7)
    ax_resid_A.axhline(0, linestyle='--', color='black')
    ax_resid_A.set_title(f"{model_name_A}\nResiduals vs Fitted", **title_style)
    ax_resid_A.set_xlabel("Fitted", fontsize=12)
    ax_resid_A.set_ylabel("Residuals", fontsize=12)
    ax_resid_A.tick_params(labelsize=10)
    ax_resid_A.grid(True, linestyle='--', linewidth=0.5)

    ax_hist_A.hist(residuals_A, bins=15, color='steelblue', alpha=0.8, edgecolor='black')
    ax_hist_A.axvline(0, color='black', linestyle='--')
    ax_hist_A.set_title("Histogram", **title_style)
    ax_hist_A.set_xlabel("Residual", fontsize=12)
    ax_hist_A.set_ylabel("Freq", fontsize=12)
    ax_hist_A.tick_params(labelsize=10)
    ax_hist_A.grid(True, linestyle='--', linewidth=0.5)
    tick_vals = np.linspace(-0.1, 0.05, 6)
    ax_hist_A.set_xlim(-0.1, 0.05)
    ax_hist_A.set_xticks(tick_vals)

    stats.probplot(norm_residuals_A, dist="norm", plot=ax_qq_A)
    ax_qq_A.set_title("Q-Q Plot", **title_style)
    ax_qq_A.set_xlabel("Theoretical", fontsize=12)
    ax_qq_A.set_ylabel("Normalized", fontsize=12)
    ax_qq_A.tick_params(labelsize=10)

    ax_resid_B.errorbar(fitted_B, residuals_B, yerr=sigma_B, fmt='o', color='red', ecolor='black', capsize=3, alpha=0.7)
    ax_resid_B.axhline(0, linestyle='--', color='black')
    ax_resid_B.set_title(f"{model_name_B}\nResiduals vs Fitted", **title_style)
    ax_resid_B.set_xlabel("Fitted", fontsize=12)
    ax_resid_B.set_ylabel("Residuals", fontsize=12)
    ax_resid_B.tick_params(labelsize=10)
    ax_resid_B.grid(True, linestyle='--', linewidth=0.5)

    ax_hist_B.hist(residuals_B, bins=15, color='salmon', alpha=0.8, edgecolor='black')
    ax_hist_B.axvline(0, color='black', linestyle='--')
    ax_hist_B.set_title("Histogram", **title_style)
    ax_hist_B.set_xlabel("Residual", fontsize=12)
    ax_hist_B.set_ylabel("Freq", fontsize=12)
    ax_hist_B.tick_params(labelsize=10)
    ax_hist_B.grid(True, linestyle='--', linewidth=0.5)
    ax_hist_B.set_xlim(-0.1, 0.05)
    ax_hist_B.set_xticks(tick_vals)

    stats.probplot(norm_residuals_B, dist="norm", plot=ax_qq_B)
    ax_qq_B.set_title("Q-Q Plot", **title_style)
    ax_qq_B.set_xlabel("Theoretical", fontsize=12)
    ax_qq_B.set_ylabel("Normalized", fontsize=12)
    ax_qq_B.tick_params(labelsize=10)

    plt.savefig(f"PsArrh_{model_name_A}_{model_name_B}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()



def plot_dual_model_with_sidepanels(dDKTB, modelA, modelB, model1, model2):
    plot_pseudo_arrhenius_dual(
        dDKTB,
        modelA["lnt"], modelB["lnt"],
        modelA["params"], modelB["params"],
        modelA["name"], modelB["name"],
        modelA["residuals"], modelA["fitted"], modelA["sigma"],
        modelB["residuals"], modelB["fitted"], modelB["sigma"]
    )



def main():
    print("Current working directory:", os.getcwd())

    global dataDurango, dataKTB
    dataDurango = load_and_process_data('DurangoDataR.txt')
    dataKTB = load_and_process_data('KTB.txt')

    appendix_KTB = pd.DataFrame({
        'time (s)': dataKTB['tempoS'],
        'temperature (K)': dataKTB['temperK'],
        'rho': dataKTB['rho'],
        'e.rho': dataKTB['e.rho']
    })
    appendix_KTB.to_csv('annealing_appendix_data_KTB.csv', index=False, float_format='%.4g')

    appendix_Durango = pd.DataFrame({
        'time (s)': dataDurango['tempoS'],
        'temperature (K)': dataDurango['temperK'],
        'rho': dataDurango['rho'],
        'e.rho': dataDurango['e.rho']
    })
    appendix_Durango.to_csv('annealing_appendix_data_DURANGO.csv', index=False, float_format='%.4g')

    models = {
        'PA': {'func': f_PA, 'initial': [5.631, 0.1865, -10.46], 'lnt': lnt_PA},
        'PC': {'func': f_PC, 'initial': [-4.910, 0.1944, -9.610], 'lnt': lnt_PC},
        'FA': {'func': f_FA, 'initial': [-8.518, 0.1266, -20.99, 0.2985], 'lnt': lnt_FA},
        'FC': {'func': f_FC, 'initial': [-9.449, 0.1627, -24.58, -0.8626], 'lnt': lnt_FC},
    }

    combined_data = pd.concat([dataDurango, dataKTB], ignore_index=True)

    results = []
    results_numeric = []
    fit_results = {}

    for name, model in models.items():
        print(f"\nFitting model: {name}")
        fit_variable = 'r' if name == 'FC' else 'rho'
        res = DAM_fit(model['func'], model['initial'], fit_variable=fit_variable)
        fit_results[name] = res

        res_df = pd.DataFrame({
            'fitted_values': res['fitted_values'],
            'residuals': res['residuals'],
            'sigma': res['sigma']
        })
        res_df.to_csv(f'residuals_{name}.csv', index=False)

        print(f"Residual analysis for model {name}:")
        analyze_residuals(res_df)

        params = list(res['params'])
        errors = list(res['errors'])
        while len(params) < 4:
            params.append(None)
            errors.append(None)

        results.append({
            'Model': name,
            '$c_0$': f"{params[0]:.4f} ({errors[0]:.4f})" if params[0] is not None else '--',
            '$c_1$': f"{params[1]:.4f} ({errors[1]:.4f})" if params[1] is not None else '--',
            '$c_2$': f"{params[2]:.4f} ({errors[2]:.4f})" if params[2] is not None else '--',
            '$c_3$': f"{params[3]:.4f} ({errors[3]:.4f})" if params[3] is not None else '--',
            'Reduced chi^2': f"{res['reduced_chi_squared']:.4f}"
        })

        results_numeric.append({
            'Model': name,
            'c_0': params[0],
            'c_0_err': errors[0],
            'c_1': params[1],
            'c_1_err': errors[1],
            'c_2': params[2],
            'c_2_err': errors[2],
            'c_3': params[3],
            'c_3_err': errors[3],
            'Reduced_chi2': res['reduced_chi_squared']
        })

    summary_df = pd.DataFrame(results)
    summary_df.to_csv('model_fit_summary.csv', index=False)
    with open('model_fit_summary.tex', 'w') as f:
        f.write(summary_df.to_latex(index=False, float_format='%.4f', na_rep='--'))

    summary_numeric_df = pd.DataFrame(results_numeric)
    summary_numeric_df.to_csv('model_fit_summary_numeric.csv', index=False)

    print("\nFinal Model Summary:")
    print(summary_df)
    print("\nFinal Numeric Model Summary:")
    print(summary_numeric_df)
    print("\nFC covariance matrix:")
    print(fit_results['FC']['covariance'])

    dual_models = {
        'Arrhenius': ('PA', 'FA'),
        'Curvilinear': ('PC', 'FC'),
    }

    for group, (model1, model2) in dual_models.items():
        modelA = {
            'name': model1,
            'lnt': models[model1]['lnt'],
            'params': fit_results[model1]['params'],
            'fitted': fit_results[model1]['fitted_values'],
            'residuals': fit_results[model1]['residuals'],
            'sigma': fit_results[model1]['sigma']
        }
        modelB = {
            'name': model2,
            'lnt': models[model2]['lnt'],
            'params': fit_results[model2]['params'],
            'fitted': fit_results[model2]['fitted_values'],
            'residuals': fit_results[model2]['residuals'],
            'sigma': fit_results[model2]['sigma']
        }
        plot_dual_model_with_sidepanels(combined_data, modelA, modelB, model1, model2)


if __name__ == '__main__':
    main()


Current working directory: /home/ging/projects/PD/Fission-Track-Annealing/notebooks


FileNotFoundError: [Errno 2] No such file or directory: 'DurangoDataR.txt'